# 自定义 relay 量化算子(python)

In [1]:
from testing import viz_expr # 可视化 relay

In [2]:
from tvm.relay.testing import run_infer_type
from tvm.relay.dataflow_pattern import (
    wildcard, is_op,
    # FunctionPattern,
    DFPatternCallback,
    rewrite
)
import tvm
from tvm.ir.attrs import DictAttrs
from tvm import relay, te, topi
from tvm.relay.op import op as _op
from tvm.target import generic_func

@generic_func
def schedule_special_op(attrs, outs, target):
    with target:
        outs = [outs] if isinstance(outs, te.tensor.Tensor) else outs
        output = outs[0]
        sch = te.create_schedule(output.op)   
        return sch

```{topic} 主题
尽可能仅仅使用 Python 实现 Relay 算子的定义。
```

In [3]:
from d2py.utils.file import mkdir
root_dir = ".temp"
mkdir(f"{root_dir}/logs")

In [4]:
import torch
from torch.nn import functional as F
from torch import nn
from torch.onnx import OperatorExportTypes, utils

class M(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 8, 1, 1, 0, bias=False, groups=1)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        x = self.conv(x)
        x = self.pool(x)
        b, c, h, w = x.shape
        x = x.view((b, h, w, c))
        x = F.softmax(x, dim=3)
        x = x.view((b, h * w * c))
        return x

model = M()
model.eval()

shape = 1, 3, 8, 8
input_name = "data"
xx = torch.rand(*shape, dtype=torch.float32, requires_grad=False)
# model = torch.jit.trace(model, xx)
# 导出模型
output_name = "test"
utils.export(
    model,               # torch 模型
    xx,                         # 模型输入或者对于多个输入，使用元组
    f"{root_dir}/{output_name}.onnx",               # 模型保存的位置（可以是文件或类似文件的对象）
    export_params=True,        # 将训练后的参数权重存储在模型文件内
    opset_version=17,          # 导出模型的 ONNX 版本
    do_constant_folding=True,  # 是否执行常量折叠以进行优化
    input_names = [input_name],    # 模型的输入名称
    output_names = ['output'], # 模型的输出名称
    keep_initializers_as_inputs=True,
    # export_modules_as_functions=True,
    verbose=True,
    operator_export_type=OperatorExportTypes.ONNX_FALLTHROUGH,
    # dynamic_axes={'data' : {0 : 'batch_size'},    # 可变长度的轴
    #               'output' : {0 : 'batch_size'}}
)

Exported graph: graph(%data : Float(1, 3, 8, 8, strides=[192, 64, 8, 1], requires_grad=0, device=cpu),
      %conv.weight : Float(8, 3, 1, 1, strides=[3, 1, 1, 1], requires_grad=1, device=cpu)):
  %/conv/Conv_output_0 : Float(1, 8, 8, 8, strides=[512, 64, 8, 1], requires_grad=0, device=cpu) = onnx::Conv[dilations=[1, 1], group=1, kernel_shape=[1, 1], pads=[0, 0, 0, 0], strides=[1, 1], onnx_name="/conv/Conv"](%data, %conv.weight), scope: __main__.M::/torch.nn.modules.conv.Conv2d::conv # /media/pc/data/tmp/cache/conda/envs/py312x/lib/python3.12/site-packages/torch/nn/modules/conv.py:456:0
  %/pool/GlobalAveragePool_output_0 : Float(1, 8, 1, 1, strides=[8, 1, 1, 1], requires_grad=1, device=cpu) = onnx::GlobalAveragePool[onnx_name="/pool/GlobalAveragePool"](%/conv/Conv_output_0), scope: __main__.M::/torch.nn.modules.pooling.AdaptiveAvgPool2d::pool # /media/pc/data/tmp/cache/conda/envs/py312x/lib/python3.12/site-packages/torch/nn/functional.py:1260:0
  %/Constant_output_0 : Long(4, strides=

前端导入：

In [5]:
import numpy as np
import onnx
import tvm
from tvm import relay

class Reshape4dSoftmaxReshape2dRewrite(DFPatternCallback):
    def __init__(self):
        super().__init__()
        self.x = wildcard()
        self.reshape4d = is_op("reshape")(self.x) # 将 NCHW 转换为 NHWC，其他 H=W=1
        self.softmax = is_op("nn.softmax")(self.reshape4d)
        self.softmax_axis = self.softmax.has_attr({"axis": 3})
        self.reshape2d = is_op("reshape")(self.softmax_axis)
        self.pattern = self.reshape2d

    def callback(self, pre, post, node_map):
        x = node_map[self.x][0]
        relay.transform.InferTypeLocal(x).shape
        x = relay.nn.softmax(x, axis=1)
        relay.transform.InferTypeLocal(x)
        x = relay.transpose(x, (0, 2, 3, 1))
        relay.transform.InferTypeLocal(x)
        x = relay.reshape(x, (1, -1))
        relay.transform.InferTypeLocal(x)
        return x
    
onnx_model = onnx.load(f"{root_dir}/{output_name}.onnx")
mod, params = relay.frontend.from_onnx(onnx_model, {input_name: shape}, freeze_params=True)
mod = relay.transform.InferType()(mod)
mod.show()
mod["main"] = rewrite(Reshape4dSoftmaxReshape2dRewrite(), mod["main"])
mod.show()

## 未成功量化 ``softmax_transpose_reshape2d`` 结构

In [6]:
from dataclasses import dataclass

@dataclass
class Dataset:
    input_name: str
    shape: tuple

    def __iter__(self):
        for _ in range(2):
            yield {self.input_name: np.random.normal(0, 1, size=self.shape).astype("float32")}

dataset = Dataset(input_name, shape)

with tvm.transform.PassContext(opt_level=3):
    with relay.quantize.qconfig(
        skip_conv_layers=[],
        calibrate_mode="kl_divergence", 
        weight_scale="max",
        # round_for_shift=True,
        # rounding="TONEAREST", # "UPWARD" or "TONEAREST"
        skip_dense_layer=False,
    ):
        qmod = relay.quantize.quantize(mod, params, dataset)
qmod.show()

## 为 `nn.softmax` 添加分区规则

原始的 `nn.global_avg_pool2d` 规则把其后所有算子均视为非量化算子：

In [7]:
quant_passes = tvm.transform.Sequential([relay.quantize.partition(), relay.quantize.annotate()])
with tvm.transform.PassContext(
    opt_level=3, 
    required_pass=["QuantizeAnnotate", "QuantizeCalibrate", "QuantizeRealize"]):
    with relay.quantize.qconfig(
        skip_conv_layers=[],
        calibrate_mode="kl_divergence", 
        weight_scale="max",
        # round_for_shift=True,
        # rounding="TONEAREST", # "UPWARD" or "TONEAREST"
        skip_dense_layer=False,
    ): 
        annotate_mod = quant_passes(mod)
annotate_mod.show()

此时无法对 `nn.softmax` 进行量化，需要重置 `nn.global_avg_pool2d` 分区规则。

In [8]:
from tvm.relay.quantize._partition import (
    register_partition_function, 
    partition_expr_check,
    QPartitionExpr
)
from tvm.relay.quantize.quantize import _forward_op

relay.op.get("nn.global_avg_pool2d").reset_attr("FQPartitionRewrite")
def pool2d_partition_function(ref_call, new_args, ctx):
    cond, expr = partition_expr_check(new_args[0])
    if cond:
        expr = new_args[0].realize()
        return QPartitionExpr(_forward_op(ref_call, [expr]))
    return None

register_partition_function("nn.global_avg_pool2d", pool2d_partition_function)

<function __main__.pool2d_partition_function(ref_call, new_args, ctx)>

In [9]:
quant_passes = tvm.transform.Sequential([relay.quantize.partition(), relay.quantize.annotate()])
with tvm.transform.PassContext(
    opt_level=3, 
    required_pass=["QuantizeAnnotate", "QuantizeCalibrate", "QuantizeRealize"]):
    with relay.quantize.qconfig(
        skip_conv_layers=[],
        calibrate_mode="kl_divergence", 
        weight_scale="max",
        # round_for_shift=True,
        # rounding="TONEAREST", # "UPWARD" or "TONEAREST"
        skip_dense_layer=False,
    ): 
        annotate_mod = quant_passes(mod)
annotate_mod.show()

In [10]:
@register_partition_function("nn.softmax")
def softmax_partition_function(ref_call, new_args, ctx):
    """Rewrite function for softmax for partition"""
    data_cond, data = partition_expr_check(new_args[0])

    if data_cond:
        data = new_args[0].realize()
    ret = _forward_op(ref_call, [data])
    return QPartitionExpr(ret)

In [16]:
quant_passes = tvm.transform.Sequential([relay.quantize.partition(), relay.quantize.annotate()])
with tvm.transform.PassContext(
    opt_level=3, 
    required_pass=["QuantizeAnnotate", "QuantizeCalibrate", "QuantizeRealize"]):
    with relay.quantize.qconfig(
        skip_conv_layers=[],
        calibrate_mode="kl_divergence", 
        weight_scale="max",
        # round_for_shift=True,
        # rounding="TONEAREST", # "UPWARD" or "TONEAREST"
        skip_dense_layer=False,
    ): 
        annotate_mod = quant_passes(mod)
annotate_mod.show()

## 为 `nn.softmax` 添加注解规则

重置 `nn.global_avg_pool2d` 注解规则

In [18]:
from tvm.relay.quantize._calibrate import calibrate

dataset = Dataset(input_name, shape)

with tvm.transform.PassContext(
    opt_level=3, 
    required_pass=["QuantizeAnnotate"]):
   
    quant_passes = tvm.transform.Sequential([
        relay.quantize.partition(), relay.quantize.annotate(), 
    ])
    with relay.quantize.qconfig(
        skip_conv_layers=[],
        calibrate_mode="kl_divergence", 
        weight_scale="max",
        # round_for_shift=True,
        # rounding="TONEAREST", # "UPWARD" or "TONEAREST"
        skip_dense_layer=False,
    ): 
        annotate_mod = quant_passes(mod)
annotate_mod.show()

In [19]:
from tvm.relay.quantize._annotate import (
    register_annotate_function,
    _get_expr_kind, QAnnotateExpr,

)
from tvm.relay.quantize.quantize import quantize_context, QAnnotateKind

def annotate_identity_rewrite(ref_call, new_args, ctx):
    """Rewrite function for avg_pool2d"""
    if quantize_context().check_to_skip(ref_call):
        return None

    expr, x_kind = _get_expr_kind(new_args[0])
    if x_kind is None:
        return None
    expr = _forward_op(ref_call, [expr])
    return QAnnotateExpr(expr, QAnnotateKind.ACTIVATION)

relay.op.get("nn.global_avg_pool2d").reset_attr("FQAnnotateRewrite")
register_annotate_function("nn.global_avg_pool2d", annotate_identity_rewrite)

<function tvm.relay.quantize._annotate.register_annotate_function.<locals>._register.<locals>.frewrite_with_guard(ref_call, new_args, ctx)>

In [20]:
from tvm.relay.quantize._calibrate import calibrate

# dataset = Dataset(input_name, shape)

with tvm.transform.PassContext(
    opt_level=3, 
    required_pass=["QuantizeAnnotate"]):
   
    quant_passes = tvm.transform.Sequential([
        relay.quantize.partition(), relay.quantize.annotate(), 
    ])
    with relay.quantize.qconfig(
        skip_conv_layers=[],
        calibrate_mode="kl_divergence", 
        weight_scale="max",
        # round_for_shift=True,
        # rounding="TONEAREST", # "UPWARD" or "TONEAREST"
        skip_dense_layer=False,
    ): 
        annotate_mod = quant_passes(mod)
annotate_mod.show()

In [21]:
from tvm.relay.quantize._annotate import attach_simulated_quantize

@register_annotate_function("nn.softmax")
def softmax_rewrite(ref_call, new_args, ctx):
    """Rewrite function for softmax. Lhs of nn.softmax will be quantized to
    input field.
    Output would be in activation field"""
    if quantize_context().check_to_skip(ref_call):
        return None

    lhs_expr, lhs_kind = _get_expr_kind(new_args[0])

    if lhs_kind is None or lhs_kind == QAnnotateKind.ACTIVATION:
        lhs_expr = attach_simulated_quantize(lhs_expr, QAnnotateKind.INPUT)

    expr = _forward_op(ref_call, [lhs_expr])

    return QAnnotateExpr(expr, QAnnotateKind.ACTIVATION)

In [23]:
from tvm.relay.quantize._calibrate import calibrate

# dataset = Dataset(input_name, shape)

with tvm.transform.PassContext(
    opt_level=3, 
    required_pass=["QuantizeAnnotate"]):
   
    quant_passes = tvm.transform.Sequential([
        relay.quantize.partition(), relay.quantize.annotate(), 
    ])
    with relay.quantize.qconfig(
        skip_conv_layers=[],
        calibrate_mode="kl_divergence", 
        weight_scale="max",
        # round_for_shift=True,
        # rounding="TONEAREST", # "UPWARD" or "TONEAREST"
        skip_dense_layer=False,
    ): 
        annotate_mod = quant_passes(mod)
annotate_mod.show()

In [25]:
from tvm.relay.quantize._calibrate import calibrate

dataset = Dataset(input_name, shape)

with tvm.transform.PassContext(opt_level=3):
   
    with relay.quantize.qconfig(
        skip_conv_layers=[],
        calibrate_mode="kl_divergence", 
        weight_scale="max",
        # round_for_shift=True,
        # rounding="TONEAREST", # "UPWARD" or "TONEAREST"
        skip_dense_layer=False,
    ): 
        qmod = relay.quantize.quantize(mod, params=params, dataset=dataset)
qmod.show()